### **Experiments on smaller amount of data**

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)

In [2]:
PAYSIM_DATA = Path("../data/raw/paysim_data.csv")

paysim_df = pd.read_csv(PAYSIM_DATA)

In [3]:
SPLIT_STEP = 500

stream_df = paysim_df[paysim_df["step"] > SPLIT_STEP].copy()
selected_accounts = (
    stream_df["nameOrig"]
    .drop_duplicates()
    .sample(n=1000, random_state=42)
)

stream_subset = stream_df[stream_df["nameOrig"].isin(selected_accounts)].copy()

print("Accounts:", stream_subset["nameOrig"].nunique())
print("Transactions:", len(stream_subset))
print("Step range:", stream_subset["step"].min(), "-", stream_subset["step"].max())

Accounts: 1000
Transactions: 1000
Step range: 502 - 732


In [4]:
account_info = (
    stream_subset
    .groupby("nameOrig")
    .agg(
        first_transaction_step=("step", "min"),
        transaction_count=("step", "size")
    )
    .reset_index()
    .rename(columns={"nameOrig": "account_id"})
)

account_info.head(10)

,account_id,first_transaction_step,transaction_count
0,C1000046384,544,1
1,C100248285,691,1
2,C1007236777,505,1
3,C1008332047,514,1
4,C100865421,544,1
5,C1012023007,691,1
6,C1012651108,542,1
7,C1015010030,684,1
8,C1017809041,598,1
9,C1023768253,597,1


In [5]:
account_info["first_transaction_step"].describe()

count    1000.000000
mean      598.162000
std        63.880392
min       502.000000
25%       544.000000
50%       590.000000
75%       664.000000
max       732.000000
Name: first_transaction_step, dtype: float64

In [6]:
print(account_info["transaction_count"].value_counts().sort_index())

print("Accounts with first transaction at step 501:", (account_info["first_transaction_step"] == 501).sum())

transaction_count
1    1000
Name: count, dtype: int64
Accounts with first transaction at step 501: 0


In [7]:
IDENTITY_DATA = Path("../data/raw/train_identity.csv")
TRANSACTION_DATA = Path("../data/raw/train_transaction.csv")

df_identity = pd.read_csv(IDENTITY_DATA)
df_transaction = pd.read_csv(TRANSACTION_DATA)

In [8]:
device_columns = [
    "DeviceType",
    "id_30",
    "id_31",
    "DeviceInfo",
    "id_33",
]

devices_df = df_identity[device_columns].copy()

devices_df.head()

,DeviceType,id_30,id_31,DeviceInfo,id_33
0,mobile,Android 7.0,samsung browser 6.2,SAMSUNG SM-G892A Build/NRD90M,2220x1080
1,mobile,iOS 11.1.2,mobile safari 11.0,iOS Device,1334x750
2,desktop,NaN,chrome 62.0,Windows,NaN
3,desktop,NaN,chrome 62.0,NaN,NaN
4,desktop,Mac OS X 10_11_6,chrome 62.0,MacOS,1280x800


In [9]:
device_profiles = (
    df_identity
    .groupby(device_columns)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

device_profiles.head(20)

,DeviceType,id_30,id_31,DeviceInfo,id_33,count
1578,desktop,Windows 10,chrome 63.0,Windows,1920x1080,2566
2263,desktop,Windows 7,chrome 63.0,Windows,1920x1080,1421
1565,desktop,Windows 10,chrome 63.0,Windows,1366x768,1390
1837,desktop,Windows 10,edge 16.0,Windows,1366x768,1249
2582,desktop,Windows 7,ie 11.0 for desktop,Trident/7.0,1920x1080,1218
4610,mobile,iOS 11.1.2,mobile safari 11.0,iOS Device,1334x750,1083
4658,mobile,iOS 11.2.1,mobile safari 11.0,iOS Device,1334x750,1069
4661,mobile,iOS 11.2.1,mobile safari 11.0,iOS Device,2208x1242,865
1850,desktop,Windows 10,edge 16.0,Windows,1920x1080,860
4613,mobile,iOS 11.1.2,mobile safari 11.0,iOS Device,2208x1242,822


In [10]:
device_profiles["probability"] = (device_profiles["count"] / device_profiles["count"].sum())   # probability of each device profile among complete identity records

device_profiles.head(20)

,DeviceType,id_30,id_31,DeviceInfo,id_33,count,probability
1578,desktop,Windows 10,chrome 63.0,Windows,1920x1080,2566,0.036080
2263,desktop,Windows 7,chrome 63.0,Windows,1920x1080,1421,0.019980
1565,desktop,Windows 10,chrome 63.0,Windows,1366x768,1390,0.019544
1837,desktop,Windows 10,edge 16.0,Windows,1366x768,1249,0.017562
2582,desktop,Windows 7,ie 11.0 for desktop,Trident/7.0,1920x1080,1218,0.017126
4610,mobile,iOS 11.1.2,mobile safari 11.0,iOS Device,1334x750,1083,0.015228
4658,mobile,iOS 11.2.1,mobile safari 11.0,iOS Device,1334x750,1069,0.015031
4661,mobile,iOS 11.2.1,mobile safari 11.0,iOS Device,2208x1242,865,0.012163
1850,desktop,Windows 10,edge 16.0,Windows,1920x1080,860,0.012092
4613,mobile,iOS 11.1.2,mobile safari 11.0,iOS Device,2208x1242,822,0.011558


In [11]:
complete_profiles = df_identity.dropna(subset=device_columns)

complete_profiles.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS
5,2987017,-5.0,61141.0,3.0,0.0,3.0,0.0,NaN,NaN,3.0,...,chrome 62.0,24.0,1366x768,match_status:2,T,F,T,T,desktop,Windows
11,2987057,0.0,88525.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,chrome 62.0 for android,32.0,1920x1080,match_status:2,T,F,T,T,mobile,SM-G930V Build/NRD90M


In [12]:
print("All identity records:", len(devices_df))
print("Complete records:", len(complete_profiles))
print("Complete records rate:", f"{len(complete_profiles) / len(devices_df):.2f}")
print("Unique complete profiles:", len(device_profiles))

All identity records: 144233
Complete records: 71120
Complete records rate: 0.49
Unique complete profiles: 4855


In [13]:
# check if the deletetion of incomplete device profiles (with at least one nan column from DeviceType, id_30, id_31, DeviceInfo, id_33) influences the distribution of devices
print("All identity records:")
print(devices_df["DeviceType"].value_counts(normalize=True, dropna=False) * 100)

print("\nComplete records:")
print(complete_profiles["DeviceType"].value_counts(normalize=True, dropna=False) * 100)

All identity records:
DeviceType
desktop    59.046820
mobile     38.579937
NaN         2.373243
Name: proportion, dtype: float64

Complete records:
DeviceType
desktop    66.117829
mobile     33.882171
Name: proportion, dtype: float64


In [14]:
devices_df["is_complete"] = devices_df[device_columns].notna().all(axis=1)

print(devices_df.groupby("DeviceType")["is_complete"].mean() * 100)

DeviceType
desktop    55.213996
mobile     43.304879
Name: is_complete, dtype: float64


In [15]:
# add probabilites for complete device profiles (with nan features too) 

# the og IEEE-CIS data proportions
device_type_probs = (
    devices_df["DeviceType"]
    .value_counts(normalize=True)
)

# probability of each profile within its device type
device_profiles["conditional_probability"] = (
    device_profiles["count"] /
    device_profiles.groupby("DeviceType")["count"].transform("sum")
)

# adjusted probability of each complete profile (in a dataset with also nan features in device profile)
device_profiles["adjusted_probability"] = (
    device_profiles["DeviceType"].map(device_type_probs) *
    device_profiles["conditional_probability"]
)

device_profiles.head(20)

,DeviceType,id_30,id_31,DeviceInfo,id_33,count,probability,conditional_probability,adjusted_probability
1578,desktop,Windows 10,chrome 63.0,Windows,1920x1080,2566,0.036080,0.054569,0.033005
2263,desktop,Windows 7,chrome 63.0,Windows,1920x1080,1421,0.019980,0.030219,0.018277
1565,desktop,Windows 10,chrome 63.0,Windows,1366x768,1390,0.019544,0.029560,0.017879
1837,desktop,Windows 10,edge 16.0,Windows,1366x768,1249,0.017562,0.026561,0.016065
2582,desktop,Windows 7,ie 11.0 for desktop,Trident/7.0,1920x1080,1218,0.017126,0.025902,0.015666
4610,mobile,iOS 11.1.2,mobile safari 11.0,iOS Device,1334x750,1083,0.015228,0.044943,0.017761
4658,mobile,iOS 11.2.1,mobile safari 11.0,iOS Device,1334x750,1069,0.015031,0.044362,0.017531
4661,mobile,iOS 11.2.1,mobile safari 11.0,iOS Device,2208x1242,865,0.012163,0.035897,0.014186
1850,desktop,Windows 10,edge 16.0,Windows,1920x1080,860,0.012092,0.018289,0.011062
4613,mobile,iOS 11.1.2,mobile safari 11.0,iOS Device,2208x1242,822,0.011558,0.034112,0.013480


In [16]:
print(device_profiles.groupby("DeviceType")["adjusted_probability"].sum())
print(device_profiles["adjusted_probability"].sum())
# no desktop domination

DeviceType
desktop    0.604822
mobile     0.395178
Name: adjusted_probability, dtype: float64
1.0


In [17]:
sampled_profiles = device_profiles.sample(
    n=len(account_info),
    weights="adjusted_probability",
    replace=True,
    random_state=42
)

In [18]:
print(sampled_profiles["DeviceType"].value_counts(normalize=True) * 100)

DeviceType
desktop    59.6
mobile     40.4
Name: proportion, dtype: float64


In [19]:
sampled_profiles.head()

,DeviceType,id_30,id_31,DeviceInfo,id_33,count,probability,conditional_probability,adjusted_probability
4793,mobile,iOS 11.3.0,mobile safari generic,iOS Device,2208x1242,248,0.003487,0.010292,0.004067
492,desktop,Mac OS X 10_11_3,safari generic,MacOS,2560x1600,2,0.000028,0.000043,0.000026
400,desktop,Mac OS X 10_10_5,chrome 63.0,MacOS,1920x1080,34,0.000478,0.000723,0.000437
4492,mobile,iOS 10.3.3,mobile safari generic,iOS Device,1334x750,89,0.001251,0.003693,0.001460
1850,desktop,Windows 10,edge 16.0,Windows,1920x1080,860,0.012092,0.018289,0.011062


In [20]:
sampled_profiles = sampled_profiles.reset_index(drop=True)
sampled_profiles.head()

,DeviceType,id_30,id_31,DeviceInfo,id_33,count,probability,conditional_probability,adjusted_probability
0,mobile,iOS 11.3.0,mobile safari generic,iOS Device,2208x1242,248,0.003487,0.010292,0.004067
1,desktop,Mac OS X 10_11_3,safari generic,MacOS,2560x1600,2,0.000028,0.000043,0.000026
2,desktop,Mac OS X 10_10_5,chrome 63.0,MacOS,1920x1080,34,0.000478,0.000723,0.000437
3,mobile,iOS 10.3.3,mobile safari generic,iOS Device,1334x750,89,0.001251,0.003693,0.001460
4,desktop,Windows 10,edge 16.0,Windows,1920x1080,860,0.012092,0.018289,0.011062


Add device profiles for each account

In [21]:
accounts_with_devices = pd.concat([
    account_info,
    sampled_profiles[device_columns]
],
axis=1
)

accounts_with_devices.head()

,account_id,first_transaction_step,transaction_count,DeviceType,id_30,id_31,DeviceInfo,id_33
0,C1000046384,544,1,mobile,iOS 11.3.0,mobile safari generic,iOS Device,2208x1242
1,C100248285,691,1,desktop,Mac OS X 10_11_3,safari generic,MacOS,2560x1600
2,C1007236777,505,1,desktop,Mac OS X 10_10_5,chrome 63.0,MacOS,1920x1080
3,C1008332047,514,1,mobile,iOS 10.3.3,mobile safari generic,iOS Device,1334x750
4,C100865421,544,1,desktop,Windows 10,edge 16.0,Windows,1920x1080


In [22]:
accounts_with_devices["device_id"] = [f"D{i:06d}" for i in range(1, len(accounts_with_devices) + 1)]
accounts_with_devices.head()

,account_id,first_transaction_step,transaction_count,DeviceType,id_30,id_31,DeviceInfo,id_33,device_id
0,C1000046384,544,1,mobile,iOS 11.3.0,mobile safari generic,iOS Device,2208x1242,D000001
1,C100248285,691,1,desktop,Mac OS X 10_11_3,safari generic,MacOS,2560x1600,D000002
2,C1007236777,505,1,desktop,Mac OS X 10_10_5,chrome 63.0,MacOS,1920x1080,D000003
3,C1008332047,514,1,mobile,iOS 10.3.3,mobile safari generic,iOS Device,1334x750,D000004
4,C100865421,544,1,desktop,Windows 10,edge 16.0,Windows,1920x1080,D000005


In [23]:
print("Accounts:", accounts_with_devices["account_id"].nunique())
print("Devices:", accounts_with_devices["device_id"].nunique())

Accounts: 1000
Devices: 1000


Add logging events for each account - for some close to transaction for others further from the transaction (randomly)<br><br>
Planned scenarios:
1. SHORT ACTIVITY - logging events max 6 hours before transaction and the last logging occurs between 1-30 minutes before the transaction hour<br>
The rest logins are generated earlier depending on a scenerio:
- NORMAL LOGIN: one login success  between 1-30 minutes before the transaction hour
- SINGLE_FAILURE: two attempts 15-120 seconds apart
- MULTIPLE_FAILURES - four attempts 10-60 seconds apart
- MULTIPLE_LOGINS - two login successes max 3 hours apart
2. MIXED ACTIVITY
- with p=0.8 - the same as SHORT ACTIVITY
- with p=0.2 - potentially way earlier (logging events)

In [24]:
accounts_with_devices["activity_pattern"] = np.random.choice(
    ["SHORT_ACTIVITY", "MIXED_ACTIVITY"],
    size=len(accounts_with_devices),
    p=[0.7, 0.3]
)

In [25]:
accounts_with_devices["activity_pattern"].value_counts()

activity_pattern
SHORT_ACTIVITY    712
MIXED_ACTIVITY    288
Name: count, dtype: int64

In [26]:
login_scenarios = {
    "NORMAL_LOGIN": ["LOGIN_SUCCESS"],
    "SINGLE_FAILURE": ["LOGIN_FAILED", "LOGIN_SUCCESS"],
    "MULTIPLE_FAILURES": [
        "LOGIN_FAILED",
        "LOGIN_FAILED",
        "LOGIN_FAILED",
        "LOGIN_SUCCESS"
    ],
    "MULTIPLE_LOGINS": ["LOGIN_SUCCESS", "LOGIN_SUCCESS"]
}

In [27]:
accounts_with_devices["login_scenario"] = np.random.choice(
    list(login_scenarios.keys()),
    size=len(accounts_with_devices),
    p=[0.65, 0.20, 0.05, 0.10]
)

accounts_with_devices["n_events"] = (accounts_with_devices["login_scenario"].map(lambda scenario: len(login_scenarios[scenario])))
accounts_with_devices[["account_id", "activity_pattern", "login_scenario", "n_events"]].head(10)

,account_id,activity_pattern,login_scenario,n_events
0,C1000046384,SHORT_ACTIVITY,NORMAL_LOGIN,1
1,C100248285,MIXED_ACTIVITY,NORMAL_LOGIN,1
2,C1007236777,MIXED_ACTIVITY,MULTIPLE_FAILURES,4
3,C1008332047,SHORT_ACTIVITY,SINGLE_FAILURE,2
4,C100865421,SHORT_ACTIVITY,SINGLE_FAILURE,2
5,C1012023007,SHORT_ACTIVITY,SINGLE_FAILURE,2
6,C1012651108,SHORT_ACTIVITY,SINGLE_FAILURE,2
7,C1015010030,MIXED_ACTIVITY,SINGLE_FAILURE,2
8,C1017809041,SHORT_ACTIVITY,NORMAL_LOGIN,1
9,C1023768253,MIXED_ACTIVITY,NORMAL_LOGIN,1


In [ ]:
events = []

for _, account in accounts_with_devices.iterrows():
    first_step = int(account["first_transaction_step"])
    scenario = account["login_scenario"]
    event_types = login_scenarios[scenario]

    transaction_start = (first_step - 1) * 3600
    latest_event_time = transaction_start - 1
    earliest_event_time = SPLIT_STEP * 3600

    near_start = max(earliest_event_time, transaction_start - 6 * 3600)

    if account["activity_pattern"] == "SHORT_ACTIVITY":
        sequence_window_start = near_start

    else:
        is_near = np.random.choice([True, False], p=[0.8, 0.2])

        if is_near or near_start == earliest_event_time:
            sequence_window_start = near_start
        else:
            sequence_window_start = earliest_event_time

    last_login_time = transaction_start - np.random.randint(60, 1801)

    if sequence_window_start < near_start:
        last_login_time = np.random.randint(
            sequence_window_start,
            near_start
        )

    if scenario == "NORMAL_LOGIN":
        event_times = [last_login_time]

    elif scenario == "SINGLE_FAILURE":
        gap = np.random.randint(15, 121)

        event_times = [last_login_time - gap, last_login_time]    

    elif scenario == "MULTIPLE_FAILURES":
        gaps = np.random.randint(10, 61, size=3)

        event_times = [
            last_login_time - gaps.sum(),
            last_login_time - gaps[1:].sum(),
            last_login_time - gaps[2],
            last_login_time
        ]

    else:  # MULTIPLE_LOGINS
        gap = np.random.randint(1, min(3 * 3600, last_login_time - sequence_window_start) + 1)
        event_times = [last_login_time - gap, last_login_time]

    if event_times[0] < sequence_window_start:   # keep sequence inside the selected window
        shift = sequence_window_start - event_times[0]
        event_times = [t + shift for t in event_times]

    for event_order, (event_time, event_type) in enumerate(zip(event_times, event_types)):
        events.append({
            "account_id": account["account_id"],
            "device_id": account["device_id"],
            "activity_pattern": account["activity_pattern"],
            "login_scenario": scenario,
            "event_type": event_type,
            "event_order": event_order,
            "event_time": event_time
        })

account_events = pd.DataFrame(events)

account_events["step"] = (account_events["event_time"] // 3600 + 1)
account_events["second_offset"] = (account_events["event_time"] % 3600)
account_events = (
    account_events
    .sort_values(["event_time", "account_id", "event_order"])
    .reset_index(drop=True)
)

account_events.head(10)

,account_id,device_id,activity_pattern,login_scenario,event_type,event_order,event_time,step,second_offset
0,C2096445087,D000561,MIXED_ACTIVITY,NORMAL_LOGIN,LOGIN_SUCCESS,0,1802204,501,2204
1,C1046880557,D000024,SHORT_ACTIVITY,MULTIPLE_LOGINS,LOGIN_SUCCESS,0,1802671,501,2671
2,C1046880557,D000024,SHORT_ACTIVITY,MULTIPLE_LOGINS,LOGIN_SUCCESS,1,1803096,501,3096
3,C1421199584,D000220,MIXED_ACTIVITY,NORMAL_LOGIN,LOGIN_SUCCESS,0,1803492,501,3492
4,C546468496,D000779,MIXED_ACTIVITY,NORMAL_LOGIN,LOGIN_SUCCESS,0,1804173,502,573
5,C1942492932,D000482,SHORT_ACTIVITY,NORMAL_LOGIN,LOGIN_SUCCESS,0,1805419,502,1819
6,C262764052,D000620,MIXED_ACTIVITY,SINGLE_FAILURE,LOGIN_FAILED,0,1805426,502,1826
7,C262764052,D000620,MIXED_ACTIVITY,SINGLE_FAILURE,LOGIN_SUCCESS,1,1805451,502,1851
8,C587029702,D000796,MIXED_ACTIVITY,NORMAL_LOGIN,LOGIN_SUCCESS,0,1805659,502,2059
9,C606866218,D000814,MIXED_ACTIVITY,NORMAL_LOGIN,LOGIN_SUCCESS,0,1806084,502,2484


In [29]:
accounts_with_devices["login_scenario"].value_counts(normalize=True) * 100

login_scenario
NORMAL_LOGIN         63.0
SINGLE_FAILURE       21.8
MULTIPLE_LOGINS       9.7
MULTIPLE_FAILURES     5.5
Name: proportion, dtype: float64

In [30]:
example_account = accounts_with_devices[accounts_with_devices["login_scenario"] == "MULTIPLE_FAILURES"].iloc[0]["account_id"]
account_events[account_events["account_id"] == example_account][["account_id", "step", "second_offset", "event_type"]]

,account_id,step,second_offset,event_type
23,C1007236777,504,2545,LOGIN_FAILED
24,C1007236777,504,2570,LOGIN_FAILED
25,C1007236777,504,2595,LOGIN_FAILED
26,C1007236777,504,2635,LOGIN_SUCCESS


In [33]:
device_columns = ["DeviceType", "id_30", "id_31", "DeviceInfo", "id_33"]

generated_events = account_events.merge(
    accounts_with_devices[["account_id"] + device_columns],
    on="account_id",
    how="left",
)

generated_events["event_id"] = [f"E{i:07d}" for i in range(1, len(generated_events) + 1)]
generated_events = generated_events[[
    "event_id",
    "account_id",
    "device_id",
    "event_type",
    "step",
    "second_offset",
    "event_time",
    "DeviceType",
    "id_30",
    "id_31",
    "DeviceInfo",
    "id_33"
    ]
].copy()

generated_events = generated_events.rename(columns={
    "DeviceType": "device_type",
    "id_30": "operating_system",
    "id_31": "browser",
    "DeviceInfo": "device_info",
    "id_33": "screen_resolution"
})

generated_events.head(10)

,event_id,account_id,device_id,event_type,step,second_offset,event_time,device_type,operating_system,browser,device_info,screen_resolution
0,E0000001,C2096445087,D000561,LOGIN_SUCCESS,501,2204,1802204,desktop,Mac OS X 10_13_2,chrome 63.0,MacOS,1280x800
1,E0000002,C1046880557,D000024,LOGIN_SUCCESS,501,2671,1802671,desktop,Windows 10,edge 16.0,Windows,1366x767
2,E0000003,C1046880557,D000024,LOGIN_SUCCESS,501,3096,1803096,desktop,Windows 10,edge 16.0,Windows,1366x767
3,E0000004,C1421199584,D000220,LOGIN_SUCCESS,501,3492,1803492,desktop,Windows 7,ie 11.0 for desktop,Trident/7.0,1919x1079
4,E0000005,C546468496,D000779,LOGIN_SUCCESS,502,573,1804173,desktop,Mac OS X 10_12_6,safari generic,MacOS,1366x768
5,E0000006,C1942492932,D000482,LOGIN_SUCCESS,502,1819,1805419,mobile,iOS 11.1.1,mobile safari generic,iOS Device,1334x750
6,E0000007,C262764052,D000620,LOGIN_FAILED,502,1826,1805426,mobile,iOS 11.2.5,mobile safari 11.0,iOS Device,2436x1125
7,E0000008,C262764052,D000620,LOGIN_SUCCESS,502,1851,1805451,mobile,iOS 11.2.5,mobile safari 11.0,iOS Device,2436x1125
8,E0000009,C587029702,D000796,LOGIN_SUCCESS,502,2059,1805659,mobile,Android 7.0,chrome 66.0 for android,SM-G930V Build/NRD90M,1920x1080
9,E0000010,C606866218,D000814,LOGIN_SUCCESS,502,2484,1806084,desktop,Windows 7,ie 11.0 for desktop,Trident/7.0,1296x774


In [40]:
example_account = generated_events["account_id"].iloc[2]
account_logins = generated_events[generated_events["account_id"] == example_account].sort_values("event_time")
account_transactions = stream_subset[stream_subset["nameOrig"] == example_account].sort_values("step")

print("Account:", example_account)

print("\nLOGIN EVENTS:")
display(account_logins)

print("\nTRANSACTIONS:")
display(account_transactions)

Account: C1046880557

LOGIN EVENTS:


,event_id,account_id,device_id,event_type,step,second_offset,event_time,device_type,operating_system,browser,device_info,screen_resolution
1,E0000002,C1046880557,D000024,LOGIN_SUCCESS,501,2671,1802671,desktop,Windows 10,edge 16.0,Windows,1366x767
2,E0000003,C1046880557,D000024,LOGIN_SUCCESS,501,3096,1803096,desktop,Windows 10,edge 16.0,Windows,1366x767



TRANSACTIONS:


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
6062061,502,CASH_IN,65654.57,C1046880557,6735556.55,6801211.11,C1367149659,333904.47,268249.9,0,0
